In [2]:
import pandas as pd
import sqlite3

df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

conn = sqlite3.connect('mystore.db')

df.to_sql('store_data', conn, if_exists='replace', index=False)

print("Table columns:")
cursor = conn.cursor()
cursor.execute("PRAGMA table_info(store_data);")
for column in cursor.fetchall():
    print(column[1], "-", column[2])

print("\nFirst 3 rows of data:")
sample_data = pd.read_sql_query("SELECT * FROM store_data LIMIT 3;", conn)
print(sample_data)

Table columns:
Row ID - INTEGER
Order ID - TEXT
Order Date - TEXT
Ship Date - TEXT
Ship Mode - TEXT
Customer ID - TEXT
Customer Name - TEXT
Segment - TEXT
Country - TEXT
City - TEXT
State - TEXT
Postal Code - INTEGER
Region - TEXT
Product ID - TEXT
Category - TEXT
Sub-Category - TEXT
Product Name - TEXT
Sales - REAL
Quantity - INTEGER
Discount - REAL
Profit - REAL

First 3 rows of data:
   Row ID        Order ID Order Date   Ship Date     Ship Mode Customer ID  \
0       1  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   
1       2  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   
2       3  CA-2016-138688  6/12/2016   6/16/2016  Second Class    DV-13045   

     Customer Name    Segment        Country         City  ... Postal Code  \
0      Claire Gute   Consumer  United States    Henderson  ...       42420   
1      Claire Gute   Consumer  United States    Henderson  ...       42420   
2  Darrin Van Huff  Corporate  United States  Los Angeles  ... 

In [3]:
print("Technology Products:")
tech_query = "SELECT [Order ID], [Customer Name], Category, Sales FROM store_data WHERE Category = 'Technology' LIMIT 5;"
tech_data = pd.read_sql_query(tech_query, conn)
print(tech_data)

print("\nSummary by Category:")
summary_query = """
SELECT Category,
       SUM(Sales) AS TotalSales,
       SUM(Quantity) AS TotalQuantity,
       AVG(Profit) AS AverageProfit
FROM store_data
GROUP BY Category;
"""
category_summary = pd.read_sql_query(summary_query, conn)
print(category_summary)

Technology Products:
         Order ID       Customer Name    Category     Sales
0  CA-2014-115812     Brosina Hoffman  Technology   907.152
1  CA-2014-115812     Brosina Hoffman  Technology   911.424
2  CA-2014-143336  Zuschuss Donatelli  Technology   213.480
3  CA-2016-121755       Eric Hoffmann  Technology    90.570
4  CA-2016-117590           Gene Hale  Technology  1097.544

Summary by Category:
          Category   TotalSales  TotalQuantity  AverageProfit
0        Furniture  741999.7953           8028       8.699327
1  Office Supplies  719047.0320          22906      20.327050
2       Technology  836154.0330           6939      78.752002


In [4]:
print("Top 5 Products:")
top_products_query = """
SELECT [Product Name], SUM(Sales) AS TotalSales
FROM store_data
GROUP BY [Product Name]
ORDER BY TotalSales DESC
LIMIT 5;
"""
top_products = pd.read_sql_query(top_products_query, conn)
print(top_products)

print("\nProfit by Region:")
region_query = """
SELECT Region, SUM(Profit) AS TotalProfit
FROM store_data
GROUP BY Region
ORDER BY TotalProfit DESC;
"""
region_profit = pd.read_sql_query(region_query, conn)
print(region_profit)

Top 5 Products:
                                        Product Name  TotalSales
0              Canon imageCLASS 2200 Advanced Copier   61599.824
1  Fellowes PB500 Electric Punch Plastic Comb Bin...   27453.384
2  Cisco TelePresence System EX90 Videoconferenci...   22638.480
3       HON 5400 Series Task Chairs for Big and Tall   21870.576
4         GBC DocuBind TL300 Electric Binding System   19823.479

Profit by Region:
    Region  TotalProfit
0     West  108418.4489
1     East   91522.7800
2    South   46749.4303
3  Central   39706.3625


In [5]:
print("Data validation checks:")
validation_query = """
SELECT COUNT(*) AS TotalRows,
       COUNT([Order ID]) - COUNT(DISTINCT [Order ID]) AS DuplicateOrders
FROM store_data;
"""
validation_results = pd.read_sql_query(validation_query, conn)
print(validation_results)

conn.close()

Data validation checks:
   TotalRows  DuplicateOrders
0       9994             4985
